# Jetson path — Colab GPU mock (TensorRT)

A4 speed check for the **A1 FP32 ONNX** (not locked `.pt`, not a half ONNX). TensorRT wants FP32 weights and picks per-layer FP16 at engine build (`CreateConfig(fp16=True)`). No Mac INT8 for Jetson.

Upload: `checkpoints/opt/export/prototype/onnx/yolo11s_prototype_best.onnx`

Prints A4 fields (`cold_ms`, `p50_ms`, `p95_ms`, `tile_fps`, `artifact_mb`) plus engine size and build time. Paste the markdown row into **OPTIMISATION.md** § A4 (Jetson line of the cross-target table).

Colab T4 ≠ Jetson — mock only; rebuild `.engine` on the device. Times are TRT execute (no Ultralytics letterbox/NMS).

In [4]:
!pip -q install tensorrt polygraphy onnx "nvidia-modelopt[onnx]"
import subprocess
import tensorrt as trt

gpu_name = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True
).strip()
print('gpu:', gpu_name)
print('tensorrt:', trt.__version__)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.3/59.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.8/252.8 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.0/239.0 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 21.0 MB/s eta 0:

In [5]:
from google.colab import files
from pathlib import Path
import onnx
import numpy as np

onnx_filename = "yolo11s_prototype_best.onnx"
onnx_path = Path(onnx_filename)
graph = onnx.load(str(onnx_path)).graph
inp = graph.input[0]
input_name = inp.name
input_shape = tuple(d.dim_value for d in inp.type.tensor_type.shape.dim)
IMGSZ = int(input_shape[-1])
onnx_mb = onnx_path.stat().st_size / (1024 * 1024)
print('ONNX:', onnx_path.name, f'({onnx_mb:.1f} MB)')
print('input:', input_name, input_shape, 'imgsz', IMGSZ)

ONNX: yolo11s_prototype_best.onnx (36.7 MB)
input: images (1, 3, 1280, 1280) imgsz 1280


In [6]:
import time
import onnx
from modelopt.onnx.autocast import convert_to_mixed_precision
from polygraphy.backend.trt import (
    EngineFromBytes,
    EngineFromNetwork,
    NetworkFromOnnxPath,
    SaveEngine,
    TrtRunner,
)
from polygraphy.backend.common import BytesFromPath

# TRT 11: AutoCast mixed FP16 ONNX, then build (no CreateConfig(fp16=True))
mixed_path = Path('yolo11s_prototype_best_mp.onnx')
t0 = time.perf_counter()
onnx.save(
    convert_to_mixed_precision(
        onnx_path=str(onnx_path),
        low_precision_type='fp16',
        keep_io_types=True,
    ),
    str(mixed_path),
)
autocast_s = time.perf_counter() - t0
print(f'AutoCast: {mixed_path.name}  {mixed_path.stat().st_size / (1024 * 1024):.1f} MB  autocast_s={autocast_s:.1f}')

engine_path = Path('yolo11s_prototype_best.engine')
build = EngineFromNetwork(NetworkFromOnnxPath(str(mixed_path)))
t0 = time.perf_counter()
SaveEngine(build, path=str(engine_path))()
build_s = time.perf_counter() - t0
engine_mb = engine_path.stat().st_size / (1024 * 1024)
print(f'TensorRT engine: {engine_path.name}  {engine_mb:.1f} MB  build_s={build_s:.1f}')

2026-08-26 13:30:10,569 - [modelopt][onnx] - WARNING - Shared constants were detected and duplicated accordingly.


2026-08-26 13:30:10,966 - [modelopt][onnx] - INFO - Successfully enabled 1 EPs for ORT: ['CPUExecutionProvider']


INFO:modelopt.onnx:Successfully enabled 1 EPs for ORT: ['CPUExecutionProvider']


2026-08-26 13:30:11,308 - [modelopt][onnx] - INFO - Running ONNX Runtime to obtain reference outputs (this may take a while)...


INFO:modelopt.onnx.autocast:Running ONNX Runtime to obtain reference outputs (this may take a while)...


...

INFO:modelopt.onnx.autocast:Skipping node /model.23/Concat_3: reference IO out of range: min=2.356471061706543, max=1276.886962890625, absmax=1276.886962890625, range=[-512, 512]


2026-08-26 13:30:17,984 - [modelopt][onnx] - WARNING - Did not find  in value info map! Assuming not castable


2026-08-26 13:30:19,787 - [modelopt][onnx] - INFO - Converted 316/318 nodes (99.37%) to fp16


INFO:modelopt.onnx.autocast:Converted 316/318 nodes (99.37%) to fp16


AutoCast: yolo11s_prototype_best_mp.onnx  18.5 MB  autocast_s=9.5
[I] TF32 is disabled by default. Turn on TF32 for better performance with minor accuracy differences.
[I] Configuring with profiles:[
        Profile 0:
            {images [min=[1, 3, 1280, 1280], opt=[1, 3, 1280, 1280], max=[1, 3, 1280, 1280]]}
    ]
[I] Building engine with configuration:
    Flags                  | [STRONGLY_TYPED]
    Engine Capability      | EngineCapability.STANDARD
    Memory Pools           | [WORKSPACE: 14912.69 MiB, TACTIC_DRAM: 14912.69 MiB, TACTIC_SHARED_MEMORY: 1024.00 MiB]
    Tactic Sources         | [EDGE_MASK_CONVOLUTIONS, JIT_CONVOLUTIONS]
    Profiling Verbosity    | ProfilingVerbosity.DETAILED
[I] Finished engine building in 164.040 seconds
[I] Saving engine to yolo11s_prototype_best.engine
TensorRT engine: yolo11s_prototype_best.engine  317.5 MB  build_s=165.1


In [7]:
import statistics
from datetime import datetime, timezone

WARMUP, N = 20, 100

def run_bench(feeds, warmup=WARMUP, n=N):
    times = []
    cold = None
    with TrtRunner(EngineFromBytes(BytesFromPath(str(engine_path)))) as runner:
        for i in range(warmup + n):
            feed = feeds[i % len(feeds)]
            t0 = time.perf_counter()
            runner.infer(feed)
            ms = (time.perf_counter() - t0) * 1000.0
            if cold is None:
                cold = ms
            if i >= warmup:
                times.append(ms)
    times.sort()
    p50 = times[len(times) // 2]
    p95 = times[int(round(0.95 * (len(times) - 1)))]
    return {
        'cold_ms': cold,
        'p50_ms': p50,
        'p95_ms': p95,
        'mean_ms': statistics.mean(times),
        'tile_fps': 1000.0 / p50,
        'frames': len(times),
        'warmup': warmup,
    }

def report(metrics, source):
    payload = {
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'platform': 'Colab',
        'gpu': gpu_name,
        'tensorrt': trt.__version__,
        'runtime': 'tensorrt_fp16',
        'model': onnx_path.name,
        'imgsz': IMGSZ,
        'source': source,
        'onnx_mb': round(onnx_mb, 1),
        'engine_mb': round(engine_mb, 1),
        'artifact_mb': round(onnx_mb, 1),
        'build_s': round(build_s, 1),
        'notebook': 'jetson_colab_mock.ipynb',
        **{k: (round(v, 2) if isinstance(v, float) else v) for k, v in metrics.items()},
    }
    print(
        f"gpu={gpu_name}  runtime=tensorrt_fp16  imgsz={IMGSZ}  source={source}\n"
        f"cold_ms={metrics['cold_ms']:.1f}  p50_ms={metrics['p50_ms']:.1f}  "
        f"p95_ms={metrics['p95_ms']:.1f}  mean_ms={metrics['mean_ms']:.1f}\n"
        f"tile_fps@p50={metrics['tile_fps']:.2f}  frames={metrics['frames']}  warmup={metrics['warmup']}\n"
        f"artifact_mb(onnx)={onnx_mb:.1f}  engine_mb={engine_mb:.1f}  build_s={build_s:.1f}"
    )
    print('OPTIMISATION.md § A4 row:')
    print(
        f"| **Jetson (Colab mock)** | TensorRT FP16 | {metrics['cold_ms']:.1f} | "
        f"{metrics['p50_ms']:.1f} | {metrics['p95_ms']:.1f} | {metrics['tile_fps']:.2f} | "
        f"{onnx_mb:.1f} | Colab {gpu_name}, {source} |"
    )
    return payload

dummy = [{input_name: np.zeros(input_shape, dtype=np.float32)}]
payload = report(run_bench(dummy), source='dummy')

[I] Loading bytes from yolo11s_prototype_best.engine
gpu=Tesla T4  runtime=tensorrt_fp16  imgsz=1280  source=dummy
cold_ms=19.9  p50_ms=12.6  p95_ms=13.4  mean_ms=12.7
tile_fps@p50=79.20  frames=100  warmup=20
artifact_mb(onnx)=36.7  engine_mb=317.5  build_s=165.1
OPTIMISATION.md § A4 row:
| **Jetson (Colab mock)** | TensorRT FP16 | 19.9 | 12.6 | 13.4 | 79.20 | 36.7 | Colab Tesla T4, dummy |


Optional — closer to Pi/Android: upload pack JPEGs from `data/datasets/eval_manual/images/` (resize to imgsz, RGB /255, not letterbox). Skip if dummy row is enough.

In [8]:
import cv2
from pathlib import Path
import numpy as np

# Шлях до папки з зображеннями
images_dir = Path('images')

# Знаходимо всі картинки всередині папки images
image_extensions = ('*.jpg', '*.jpeg', '*.png')
image_paths = []
for ext in image_extensions:
    image_paths.extend(images_dir.glob(ext))

feeds = []
tile_up = {}
h, w = int(input_shape[2]), int(input_shape[3])

for img_path in image_paths:
    name = img_path.name
    # Читаємо файл через повний шлях до нього (наприклад, images/tile1.jpg)
    bgr = cv2.imread(str(img_path))
    if bgr is None:
        continue

    tile_up[name] = True
    rgb = cv2.cvtColor(cv2.resize(bgr, (w, h)), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    feeds.append({input_name: np.transpose(rgb, (2, 0, 1))[None]})

print('tiles:', len(feeds), [n for n in tile_up])
payload = report(run_bench(feeds), source='eval_manual_tiles')

tiles: 83 ['13722965_2160_3840_30fps__000714.jpg', '266987__000089.jpg', '266987__000881.jpg', '266987__000909.jpg', '13722965_2160_3840_30fps__000277.jpg', '13722965_2160_3840_30fps__000829.jpg', '13722965_2160_3840_30fps__000875.jpg', '266987__000145.jpg', '13722965_2160_3840_30fps__000300.jpg', '13722965_2160_3840_30fps__000737.jpg', '13722965_2160_3840_30fps__000346.jpg', '266987__000433.jpg', '266987__000101.jpg', '266987__000837.jpg', '266987__000305.jpg', '266987__000017.jpg', '266987__000521.jpg', '13722965_2160_3840_30fps__000645.jpg', '13722965_2160_3840_30fps__000116.jpg', '266987__000477.jpg', '266987__000493.jpg', '13722965_2160_3840_30fps__000576.jpg', '266987__000821.jpg', '266987__000161.jpg', '13722965_2160_3840_30fps__000438.jpg', '13722965_2160_3840_30fps__000760.jpg', '13722965_2160_3840_30fps__000208.jpg', '266987__000117.jpg', '266987__000389.jpg', '266987__000045.jpg', '13722965_2160_3840_30fps__000484.jpg', '266987__000449.jpg', '13722965_2160_3840_30fps__000553